In [2]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve().parent 
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

results_dir = project_root / 'results'
results_dir.mkdir(exist_ok=True)

# Distributed models with PyTorch in RADAR

This notebook shows a more natural way to perform **distributed training** in RADAR using **PyTorch / PyTorch Lightning**, instead of parallelizing independent classical models.

Distributed training of the same model — for example splitting work across multiple GPUs.

In RADAR, the most convenient path for this is `TSFEDL`, because its wrapper `TsfedlAnomalyDetection` already uses `pytorch_lightning.Trainer`, and therefore accepts parameters such as:

- `accelerator`
- `devices`
- `strategy`
- `max_epochs`

This allows enabling distributed mode with very few changes.

To keep the notebook **simple**, we will use a `TSFEDL` model with a PyTorch backend.


In [ ]:
import os
import numpy as np
import pandas as pd
import torch

from TSFEDL.models_pytorch import OhShuLih_Forecaster
from RADAR.time_series.algorithms import tsfedl
from RADAR.time_series.time_series_utils import TimeSeriesProcessor
from RADAR.metrics_module import metric_accuracy, metric_precision, metric_recall, metric_AUC_ROC_scores

pd.set_option('display.max_columns', None)

print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Number of GPUs:', torch.cuda.device_count())
if torch.cuda.is_available():
    for idx in range(torch.cuda.device_count()):
        print(f'GPU {idx}: {torch.cuda.get_device_name(idx)}')

## Synthetic time series dataset

We will use a multivariate synthetic series so the example is self-contained and easy to reproduce.

We will generate:

- a base signal approximately sinusoidal,
- small noise,
- anomalies as artificial spikes.

We will then build time windows with `TimeSeriesProcessor`. 

In [ ]:
def make_synthetic_ts(n_train=3000, n_test=800, n_features=5, contamination=0.1, seed=42):
    rng = np.random.default_rng(seed)
    total = n_train + n_test
    time = np.arange(total)

    X = []
    for feature_idx in range(n_features):
        phase = feature_idx * 0.35
        signal = np.sin(0.03 * time + phase) + 0.25 * np.cos(0.01 * time + phase)
        signal = signal + rng.normal(0, 0.08, size=total)
        X.append(signal)

    X = np.stack(X, axis=1).astype(np.float32)
    y = np.zeros(total, dtype=int)

    n_anomalies = int(total * contamination)
    anomaly_idx = rng.choice(total, size=n_anomalies, replace=False)
    X[anomaly_idx] += rng.normal(3.5, 0.5, size=(n_anomalies, n_features)).astype(np.float32)
    y[anomaly_idx] = 1

    return X[:n_train], X[n_train:], y[:n_train], y[n_train:]


X_train, X_test, y_train, y_test = make_synthetic_ts()
print('X_train:', X_train.shape)
print('X_test :', X_test.shape)
print('Train anomalies:', int(y_train.sum()))
print('Test anomalies :', int(y_test.sum()))

In [ ]:
WINDOW_SIZE = 48
STEP_SIZE = 1
N_PRED = 1

processor = TimeSeriesProcessor(
    window_size=WINDOW_SIZE,
    step_size=STEP_SIZE,
    future_prediction=False,
    n_pred=N_PRED,
)

X_train_windows, y_train_windows, X_test_windows, y_test_windows = processor.process_train_test(
    X_train,
    y_train,
    X_test,
    y_test,
)

X_train_windows = torch.tensor(X_train_windows, dtype=torch.float32)
y_train_windows = torch.tensor(y_train_windows, dtype=torch.float32).unsqueeze(-1)
X_test_windows = torch.tensor(X_test_windows, dtype=torch.float32)

y_test_window_labels = (np.asarray(y_test_windows).sum(axis=1) > 0).astype(int)

print('X_train_windows:', tuple(X_train_windows.shape))
print('y_train_windows:', tuple(y_train_windows.shape))
print('X_test_windows :', tuple(X_test_windows.shape))
print('Test window labels:', y_test_window_labels.shape)

## PyTorch Lightning distributed configuration

The key is here: `TsfedlAnomalyDetection` forwards parameters to Lightning's `Trainer`.

We will use this logic:

- if there are **2 or more GPUs**, we try multi-GPU distributed mode;
- if there is **1 GPU**, we use single GPU;
- if there is no GPU, we run a functional demo on CPU.

> In notebooks, Lightning may require a special strategy such as `ddp_notebook`. If it fails in the environment, the same configuration can be moved to a script with `strategy='ddp'`. 

In [ ]:
def build_trainer_kwargs(max_epochs=3):
    trainer_kwargs = {
        'max_epochs': max_epochs,
        'logger': False,
        'enable_checkpointing': False,
        'enable_model_summary': False,
    }

    gpu_count = torch.cuda.device_count()

    if gpu_count >= 2:
        trainer_kwargs.update({
            'accelerator': 'gpu',
            'devices': min(2, gpu_count),
            'strategy': 'ddp_notebook',
        })
    elif gpu_count == 1:
        trainer_kwargs.update({
            'accelerator': 'gpu',
            'devices': 1,
        })
    else:
        trainer_kwargs.update({
            'accelerator': 'cpu',
            'devices': 1,
        })

    return trainer_kwargs


trainer_kwargs = build_trainer_kwargs(max_epochs=3)
trainer_kwargs

## TSFEDL model instantiation

We will use `OhShuLih_Forecaster` as the `top_module`. 

In [ ]:
top_module = OhShuLih_Forecaster(out_features=X_train.shape[1], n_pred=N_PRED)

model_kwargs = {
    'algorithm_': 'ohshulih',
    'loss': torch.nn.MSELoss(),
    'top_module': top_module,
    'in_features': WINDOW_SIZE,
    'batch_size': 64,
    **trainer_kwargs,
}

model = tsfedl.TsfedlAnomalyDetection(**model_kwargs)
model_kwargs

## Training

This cell is the important one: if there are multiple GPUs and `trainer_kwargs` has enabled multiple devices, this is where training becomes distributed.

In [ ]:
model.fit(X_train_windows, y_train_windows)

## Prediction and scoring



In [ ]:
scores = model.decision_function(X_test_windows)
preds = model.predict(X_test_windows)

scores = np.asarray(scores).ravel()
preds = np.asarray(preds).astype(int).ravel()

print('scores shape:', scores.shape)
print('preds shape :', preds.shape)
print('predicted positives:', int(preds.sum()))

In [ ]:
results = {
    'accuracy': round(metric_accuracy(y_test_window_labels, preds) / 100, 4),
    'precision': round(metric_precision(y_test_window_labels, preds), 4),
    'recall': round(metric_recall(y_test_window_labels, preds), 4),
    'roc_auc_scores': round(metric_AUC_ROC_scores(y_test_window_labels, scores), 4),
}

pd.DataFrame([results])

## Second example: GaoJunLi

We repeat the same flow using `GaoJunLi_Forecaster` as `top_module`. The distributed configuration (`trainer_kwargs`) is reused without changes.

In [ ]:
from TSFEDL.models_pytorch import GaoJunLi_Forecaster

top_module_gao = GaoJunLi_Forecaster(out_features=X_train.shape[1], n_pred=N_PRED)

model_gao_kwargs = {
    'algorithm_': 'gaojunli',
    'loss': torch.nn.MSELoss(),
    'top_module': top_module_gao,
    'in_features': WINDOW_SIZE,
    'batch_size': 64,
    **trainer_kwargs,
}

model_gao = tsfedl.TsfedlAnomalyDetection(**model_gao_kwargs)
model_gao_kwargs

### Training

In [ ]:
model_gao.fit(X_train_windows, y_train_windows)

### Prediction and scoring

In [ ]:
scores_gao = model_gao.decision_function(X_test_windows)
preds_gao = model_gao.predict(X_test_windows)

scores_gao = np.asarray(scores_gao).ravel()
preds_gao = np.asarray(preds_gao).astype(int).ravel()

print('scores shape:', scores_gao.shape)
print('preds shape :', preds_gao.shape)
print('predicted positives:', int(preds_gao.sum()))

In [ ]:
results_gao = {
    'accuracy': round(metric_accuracy(y_test_window_labels, preds_gao) / 100, 4),
    'precision': round(metric_precision(y_test_window_labels, preds_gao), 4),
    'recall': round(metric_recall(y_test_window_labels, preds_gao), 4),
    'roc_auc_scores': round(metric_AUC_ROC_scores(y_test_window_labels, scores_gao), 4),
}

pd.DataFrame([results_gao])